# Lab 6: Image Processing with OpenCV

Run each cell with **Shift + Enter**, from the top. Pictures appear right under the cell. The live camera views (Part G and the challenge) also appear under their cells. A line that starts with `!` is a Linux command, and `%%bash` makes a cell a small Linux session.


In [ ]:
from IPython.display import Image, display

def show(*names, width=320):
    """Show picture files under the cell."""
    for n in names:
        print(n)
        display(Image(filename=n, width=width))

## Part A, Step 1: Your Workspace and OpenCV
This notebook lives in the `lab6` folder inside your own folder. Check OpenCV and NumPy. The first line should show an OpenCV version (for example 4.8.0 or 4.12.0) and a NumPy version below 2. The second prints how many CUDA GPUs OpenCV can use: 1 means your OpenCV was built with CUDA and 0 means it was not. Either result is fine for this lab, because its functions run on the CPU.

In [ ]:
%%bash
pwd
python -c "import cv2, numpy; print(cv2.__version__, numpy.__version__)"
python -c "import cv2; print(cv2.cuda.getCudaEnabledDeviceCount())"

| Item | Your result |
|---|---|
| OpenCV version | |
| NumPy version | |
| CUDA devices seen by OpenCV | |

> **Note.** If you see `No module named cv2`, you are not in your own JupyterLab. Open the address your teacher gave you. Nothing needs to be installed in this lab. Call your instructor.

## Part A, Step 2: Make a Test Image
This draws a picture with a gradient background, a green box, a red circle, a yellow box, a line and some text. Find the **red circle** and note which colour is on the left of the background.

In [ ]:
import numpy as np
import cv2

h, w = 480, 640
img = np.zeros((h, w, 3), dtype=np.uint8)

# Background: a blue to dark gradient (BGR order)
for x in range(w):
    img[:, x] = (int(40 + 120 * x / w), 40, 20)

cv2.rectangle(img, (60, 80), (260, 240), (0, 200, 0), -1)      # green box
cv2.circle(img, (450, 160), 80, (0, 0, 255), -1)               # red circle
cv2.rectangle(img, (340, 300), (580, 420), (0, 220, 255), -1)  # yellow box
cv2.line(img, (30, 455), (610, 445), (255, 255, 255), 3)      # white line
cv2.putText(img, "JETSON", (70, 340), cv2.FONT_HERSHEY_SIMPLEX,
            1.8, (255, 255, 255), 4)

cv2.imwrite("test.png", img)
print("Saved test.png", img.shape)

show("test.png")

## Part A, Step 3: Read the Image and Look at the Numbers

In [ ]:
import cv2

img = cv2.imread("test.png")
if img is None:
    raise RuntimeError("Cannot read test.png. Run the make_image cell first.")

print("Shape:", img.shape)
print("Type:", img.dtype)
print("Pixel at row 100, column 100 (B, G, R):", img[100, 100])
print("Pixel at row 160, column 450 (B, G, R):", img[160, 450])

gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
print("Gray shape:", gray.shape)
cv2.imwrite("gray.png", gray)
print("Saved gray.png")

show("gray.png")

## Part B, Step 1: Plug In and Find the Camera
Plug the USB camera into a USB port on the Jetson. Linux shows a camera as a file such as `/dev/video0`. Some cameras show two entries (`video0` and `video1`). The first one is normally the picture and the second one is information about it.

In [ ]:
%%bash
ls /dev/video*
v4l2-ctl --list-devices

The camera your teacher chose for you on the teacher page is:

In [ ]:
import os
print(os.environ.get("LAB_CAMERA", "no camera chosen"))

| Item | Your result |
|---|---|
| Name of the camera | |
| Device file (for example `/dev/video0`) | |

## Part B, Step 2: See What the Camera Can Do
This lists the picture formats and sizes the camera supports. Most USB cameras offer MJPG (compressed, allows high frame rates) and YUYV (uncompressed, slower at large sizes). Note one size and the highest FPS listed beside it.

In [ ]:
%%bash
v4l2-ctl -d "${LAB_CAMERA:-/dev/video0}" --list-formats-ext | head -30

| Item | Your result |
|---|---|
| One supported size (for example 640x480) | |
| FPS listed for MJPG at that size | |

## Part B, Step 3: Capture a Picture
Point the camera at something with colours and clear shapes. The first frames are skipped because cameras are dark for a moment. If this cell says it cannot open the camera, check the USB cable and that no other program is using it.

In [ ]:
import cv2
import labview

# the camera your teacher chose for you on the teacher page
cap = labview.open_camera(size=None, mjpg=False)

print("Default size:", cap.get(cv2.CAP_PROP_FRAME_WIDTH), "x", cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
print("Default FPS :", cap.get(cv2.CAP_PROP_FPS))

# The first frames are often dark while the camera adjusts
for _ in range(15):
    ok, frame = cap.read()

ok, frame = cap.read()
if not ok:
    raise RuntimeError("The camera opened but gave no picture.")

cv2.imwrite("snapshot.jpg", frame)
print("Saved snapshot.jpg", frame.shape)
cap.release()

show("snapshot.jpg", width=480)

## Part C, Step 1: Pick Out Red With HSV
A **mask** is a black and white picture that is white where the colour is red. Red is near hue 0 and also near 180, so two ranges are joined.

In [ ]:
import cv2
import numpy as np

img = cv2.imread("test.png")
hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

b, g, r = cv2.split(img)
print("Mean of blue, green, red channels:",
      round(float(b.mean()), 1), round(float(g.mean()), 1), round(float(r.mean()), 1))

# Find the red circle using HSV. Red is near hue 0 (and near 180)
low1, high1 = np.array([0, 120, 70]), np.array([10, 255, 255])
low2, high2 = np.array([170, 120, 70]), np.array([180, 255, 255])
mask = cv2.inRange(hsv, low1, high1) | cv2.inRange(hsv, low2, high2)

print("Red pixels found:", int(cv2.countNonZero(mask)))
cv2.imwrite("mask_red.png", mask)
result = cv2.bitwise_and(img, img, mask=mask)
cv2.imwrite("only_red.png", result)
print("Saved mask_red.png and only_red.png")

show("mask_red.png", "only_red.png")

## Part C, Step 2: The RGB and BGR Mix-Up
This swaps the channel order by hand, so red and blue swap. What happened to the red circle and to the background?

In [ ]:
import cv2
img = cv2.imread("test.png")
wrong = img[:, :, ::-1]
cv2.imwrite("swapped.png", wrong)

show("test.png", "swapped.png")

## Part D, Step 1: Noise, Blur and Sharpen
The **average error** is how far a picture is from the clean one. The noise is random, so your numbers change a little each time.

In [ ]:
import cv2
import numpy as np

img = cv2.imread("test.png")

# Add noise so the filters have something to remove
noise = np.random.normal(0, 25, img.shape)
noisy = np.clip(img.astype(np.float32) + noise, 0, 255).astype(np.uint8)
cv2.imwrite("noisy.png", noisy)

box = cv2.blur(noisy, (5, 5))
gauss = cv2.GaussianBlur(noisy, (5, 5), 0)
median = cv2.medianBlur(noisy, 5)
cv2.imwrite("blur_box.png", box)
cv2.imwrite("blur_gauss.png", gauss)
cv2.imwrite("blur_median.png", median)

kernel = np.array([[0, -1, 0],
                   [-1, 5, -1],
                   [0, -1, 0]], dtype=np.float32)
sharp = cv2.filter2D(img, -1, kernel)
cv2.imwrite("sharpen.png", sharp)

def error(a):
    return float(np.mean(np.abs(a.astype(np.float32) - img.astype(np.float32))))

print("Average error compared with the clean image:")
print("  noisy  :", round(error(noisy), 1))
print("  box    :", round(error(box), 1))
print("  gauss  :", round(error(gauss), 1))
print("  median :", round(error(median), 1))

show("noisy.png", "blur_box.png", "blur_gauss.png", "blur_median.png", "sharpen.png")

## Part D, Step 2: Try a Bigger Kernel
In the cell above, change `(5, 5)` to `(15, 15)` in the Gaussian blur line (the kernel size must be odd) and run it again. What happens to the noise and to the sharpness of the text?

In [ ]:
# Your experiment here (copy the Gaussian blur line and change the kernel size)

## Part E, Step 1: Find Edges With Canny
This uses `test.png`. To use your camera picture, change the first line to `name = "snapshot.jpg"` and run it again.

In [ ]:
import cv2

name = "test.png"      # change to "snapshot.jpg" to use your camera picture
img = cv2.imread(name)
if img is None:
    raise RuntimeError(f"Cannot read {name}")

gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
blur = cv2.GaussianBlur(gray, (5, 5), 0)

for low, high in [(50, 150), (10, 30), (150, 250)]:
    edges = cv2.Canny(blur, low, high)
    out = f"edges_{low}_{high}.png"
    cv2.imwrite(out, edges)
    print(f"Canny {low}/{high}: {cv2.countNonZero(edges)} edge pixels -> {out}")

show("edges_50_150.png", "edges_10_30.png", "edges_150_250.png")

## Part E, Step 2: Thresholding

In [ ]:
import cv2

gray = cv2.imread("test.png", cv2.IMREAD_GRAYSCALE)

_, fixed = cv2.threshold(gray, 100, 255, cv2.THRESH_BINARY)
otsu_value, otsu = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
adaptive = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                 cv2.THRESH_BINARY, 31, 5)

cv2.imwrite("th_fixed.png", fixed)
cv2.imwrite("th_otsu.png", otsu)
cv2.imwrite("th_adaptive.png", adaptive)
print("Otsu chose the threshold:", otsu_value)
print("White pixels  fixed:", cv2.countNonZero(fixed),
      " otsu:", cv2.countNonZero(otsu), " adaptive:", cv2.countNonZero(adaptive))

show("th_fixed.png", "th_otsu.png", "th_adaptive.png")

## Part F: Find and Box Objects

In [ ]:
import cv2

img = cv2.imread("test.png")
hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

# Everything that is strongly coloured (not the dark background)
mask = cv2.inRange(hsv, (0, 150, 180), (180, 255, 255))
mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5)))

contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
print("Objects found:", len(contours))

out = img.copy()
for cnt in contours:
    area = cv2.contourArea(cnt)
    if area < 500:
        continue
    x, y, w, h = cv2.boundingRect(cnt)
    cv2.rectangle(out, (x, y), (x + w, y + h), (255, 0, 255), 2)
    print(f"  box at x={x}, y={y}, width={w}, height={h}, area={area:.0f}")
cv2.imwrite("contours.png", out)

show("contours.png", width=480)

## Part G, Step 1: A Live View With Filters
Run the cell. The live picture appears under it. Choose a mode in the list and watch the FPS. Press **Stop** when you are done, so the camera is free for the next cell.

The FPS here includes sending every picture to your browser, so it is lower than in a window on the Jetson screen.

In [ ]:
import cv2
import labview


def process(frame, state):
    mode = state["Mode"]
    if mode == "gray":
        shown = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    elif mode == "blur":
        shown = cv2.GaussianBlur(frame, (21, 21), 0)
    elif mode == "edges":
        shown = cv2.Canny(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY), 50, 150)
    else:
        shown = frame.copy()
    cv2.putText(shown, f"{mode}  {state['fps']:.0f} FPS", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, 255 if mode in ("gray", "edges") else (0, 255, 0), 2)
    return shown


live = labview.live(process, controls={"Mode": ["normal", "gray", "blur", "edges"]})

## Part G, Step 2: How Fast Is Each Frame Size?
Write down the power mode first (run the two cells below; the first asks for your password for `sudo`). Then run the speed cell **twice** and record the second run.

In [ ]:
import labview
labview.sudo_login()

In [ ]:
!sudo nvpmodel -q

In [ ]:
import time
import cv2
import numpy as np

sizes = [(320, 240), (640, 480), (1280, 720), (1920, 1080)]

for w, h in sizes:
    img = np.random.randint(0, 256, (h, w, 3), dtype=np.uint8)
    cv2.GaussianBlur(img, (15, 15), 0)            # warm-up
    start = time.perf_counter()
    for _ in range(20):
        cv2.GaussianBlur(img, (15, 15), 0)
    ms = (time.perf_counter() - start) / 20 * 1000
    print(f"{w} x {h}: Gaussian blur {ms:.2f} ms  ({1000 / ms:.0f} per second)")

## Challenge: Track a Coloured Object
This finds the biggest blue object in the camera view, draws a box around it and shows its centre. Hold a blue pen cap or a similar object in front of the camera. The picture shows the camera view with the box; choose **Mask** in the list to see the black and white mask instead.

Then change the two `np.array` lines to track another colour. HSV hue values are about 0 for red, 30 for yellow, 60 for green and 120 for blue. If the box jumps around or picks up the wrong thing, look at the mask. Adjust the saturation and value minimums (the 120 and 70) until only your object is white. Press **Stop** when you are done.

In [ ]:
import cv2
import numpy as np
import labview

# Blue objects. Change these numbers to track another colour.
low = np.array([100, 120, 70])
high = np.array([130, 255, 255])
kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))


def track(frame, state):
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, low, high)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if contours:
        biggest = max(contours, key=cv2.contourArea)
        if cv2.contourArea(biggest) > 800:
            x, y, w, h = cv2.boundingRect(biggest)
            cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)
            cv2.putText(frame, f"centre ({x + w // 2}, {y + h // 2})", (x, y - 8),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
    return mask if state["View"] == "Mask" else frame


tracker = labview.live(track, controls={"View": ["Camera", "Mask"]}, mjpg=False, size=None)